# Analyse de sensibilité sur κ — Modèle de Chiarella étendu

**Objectif** : Comprendre l'impact du paramètre de rappel fondamental κ sur :
- L'amplitude des bulles (mispricing P−V)
- La volatilité des rendements
- Les propriétés statistiques (skewness, kurtosis)

Tous les autres paramètres sont fixés (`MONTHLY_PARAMS`), γ est constant (baseline).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.model import ChiarellaModel, ModelParams
from src.simulation import MONTHLY_PARAMS
from src.analysis import compute_returns, return_statistics

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 2024

## 1. Paramètres du sweep

In [ ]:
KAPPA_VALUES = [0.01, 0.02, 0.05, 0.08, 0.15, 0.30, 0.50]

# Paramètres de base (on va juste changer kappa à chaque itération)
BASE_PARAMS = MONTHLY_PARAMS
print(f"Paramètres de base : {BASE_PARAMS}")
print(f"Valeurs de κ testées : {KAPPA_VALUES}")

## 2. Simulation pour chaque valeur de κ

In [ ]:
from dataclasses import replace

records = []

for kappa in KAPPA_VALUES:
    params = replace(BASE_PARAMS, kappa=kappa)
    model = ChiarellaModel(params=params, seed=SEED)
    res = model.simulate(n_paths=1)

    mispricing = res.P - res.V
    returns = compute_returns(res.P)
    stats = return_statistics(returns, annualization_factor=np.sqrt(12))  # mensuel → annuel

    records.append({
        'kappa':        kappa,
        'std_mispricing': float(np.std(mispricing)),
        'max_mispricing': float(np.max(np.abs(mispricing))),
        'vol_annualisee': stats['vol'],
        'skewness':       stats['skewness'],
        'kurtosis_exc':   stats['kurt_excess'],
        'jb_pval':        stats['jarque_bera_pval'],
        # On garde les trajectoires pour les graphiques
        '_t':   res.t,
        '_P':   res.P,
        '_V':   res.V,
    })

df_stats = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in records])
df_stats

## 3. Impact de κ sur le mispricing et la volatilité

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Std(P-V) ---
axes[0].plot(df_stats['kappa'], df_stats['std_mispricing'], 'o-', color='steelblue')
axes[0].axvline(0.08, color='gray', linestyle='--', alpha=0.6, label='κ calibré (0.08)')
axes[0].set_xlabel('κ')
axes[0].set_ylabel('Std(P − V)')
axes[0].set_title('Amplitude moyenne des bulles')
axes[0].legend()

# --- Max|P-V| ---
axes[1].plot(df_stats['kappa'], df_stats['max_mispricing'], 'o-', color='tomato')
axes[1].axvline(0.08, color='gray', linestyle='--', alpha=0.6, label='κ calibré (0.08)')
axes[1].set_xlabel('κ')
axes[1].set_ylabel('Max|P − V|')
axes[1].set_title('Bulle maximale observée')
axes[1].legend()

# --- Vol annualisée ---
axes[2].plot(df_stats['kappa'], df_stats['vol_annualisee'], 'o-', color='seagreen')
axes[2].axvline(0.08, color='gray', linestyle='--', alpha=0.6, label='κ calibré (0.08)')
axes[2].set_xlabel('κ')
axes[2].set_ylabel('Volatilité annualisée')
axes[2].set_title('Vol des rendements mensuels')
axes[2].legend()

fig.suptitle('Sensibilité du modèle de Chiarella à κ', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Tableau récapitulatif

In [ ]:
df_display = df_stats[['kappa', 'std_mispricing', 'max_mispricing', 'vol_annualisee', 'skewness', 'kurtosis_exc']].copy()
df_display.columns = ['κ', 'Std(P−V)', 'Max|P−V|', 'Vol annualisée', 'Skewness', 'Kurtosis excéd.']

# Variation relative par rapport à κ = 0.08 (baseline)
baseline_row = df_display[df_display['κ'] == 0.08].iloc[0]
for col in ['Std(P−V)', 'Max|P−V|', 'Vol annualisée']:
    df_display[f'Δ {col}'] = ((df_display[col] - baseline_row[col]) / baseline_row[col] * 100).round(1).astype(str) + ' %'

df_display.round(4)

## 5. Trajectoires P vs V pour κ extrêmes

In [ ]:
# On affiche κ faible, calibré, fort
kappa_showcase = [0.01, 0.08, 0.50]
showcase_records = [r for r in records if r['kappa'] in kappa_showcase]

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)

for ax, rec in zip(axes, showcase_records):
    t = rec['_t']
    ax.plot(t, rec['_P'], label='P (log-prix)', linewidth=0.8)
    ax.plot(t, rec['_V'], label='V (fondamental)', linewidth=0.8, linestyle='--')
    ax.fill_between(t, rec['_P'], rec['_V'], alpha=0.15, color='red', label='Mispricing')
    ax.set_xlabel('Temps (mois)')
    ax.set_ylabel('Log-prix / Log-valeur')
    ax.set_title(f'κ = {rec["kappa"]}')
    ax.legend(fontsize=8)

fig.suptitle('Trajectoires P vs V pour κ faible, calibré, fort', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Distribution du mispricing selon κ

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = sns.color_palette('coolwarm', len(records))

for rec, color in zip(records, colors):
    mispricing = rec['_P'] - rec['_V']
    sns.kdeplot(mispricing, ax=ax, label=f"κ={rec['kappa']}", color=color, linewidth=1.4)

ax.axvline(0, color='black', linestyle='--', alpha=0.4)
ax.set_xlabel('P − V (mispricing)')
ax.set_ylabel('Densité')
ax.set_title('Distribution du mispricing selon κ')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Conclusions

À compléter après exécution du notebook avec les résultats numériques.